<a href="https://colab.research.google.com/github/LCaravaggio/scrapers/blob/master/Google_Earth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from selenium import webdriver

options = webdriver.ChromeOptions()
options.add_argument("--window-size=2920,3220")  # Establecer resolución más alta

options.add_argument("--headless")  # Opcional: ejecuta en modo invisible

In [7]:
import pandas as pd
from PIL import Image
ciudades=pd.read_csv('Gini con latalon OECD.csv')

In [36]:
def take_image40K(X):
  driver = webdriver.Chrome(options=options)
  lat=str(ciudades.lat[X])
  lon=str(ciudades.lon[X])
  # Cargar la página
  url = f"https://earth.google.com/web/@{lat},{lon},32.10687273a,2587105.99190176d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
  driver.get(url)
  start_time = time.time()
  while time.time() - start_time < 120:
      pass
  # Tomar captura de pantalla
  screenshot_name = ciudades.Ciudad[X]+" - 40K.png"
  screenshot_name=screenshot_name.replace(":","_").replace("/",".")
  driver.save_screenshot(screenshot_name)

  img = Image.open(screenshot_name)
  width, height = img.size
  crop_top = 200
  crop_bottom = 100
  cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
  cropped_img.save(screenshot_name)

  print(f"Screenshot saved to: {screenshot_name}")
  driver.quit()

In [34]:
import math
import time
from PIL import Image
from selenium import webdriver

def take_mosaic(X, zoom_deg_lat=0.495, zoom_deg_lon=None):
    driver = webdriver.Chrome(options=options)
    
    lat_center = float(ciudades.lat[X])
    lon_center = float(ciudades.lon[X])
    city = ciudades.Ciudad[X].replace(":", "_").replace("/", ".")

    # calcular delta de longitud según latitud
    if zoom_deg_lon is None:
        zoom_deg_lon = 0.495 / math.cos(math.radians(lat_center))

    positions = []
    for dy in range(-2, 3):  # -2, -1, 0, 1, 2
        for dx in range(-2, 3):
            lat = lat_center + dy * zoom_deg_lat
            lon = lon_center + dx * zoom_deg_lon
            positions.append((dy + 2, dx + 2, lat, lon))  # guardamos fila, columna

    images = [[None for _ in range(5)] for _ in range(5)]

    for row, col, lat, lon in positions:
        url = f"https://earth.google.com/web/@{lat},{lon},10a,55000d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
        driver.get(url)
        print(f"Cargando {lat:.4f}, {lon:.4f}...")
        time.sleep(10)  # Esperar a que cargue la vista

        filename = f"{city}_r{row}_c{col}.png"
        driver.save_screenshot(filename)

        img = Image.open(filename)
        width, height = img.size
        cropped_img = img.crop((0, 200, width, height - 100))  # recorta márgenes
        images[row][col] = cropped_img

    driver.quit()

    # Composición en mosaico
    tile_width, tile_height = images[0][0].size
    mosaic = Image.new('RGB', (tile_width * 5, tile_height * 5))

    for row in range(5):
        for col in range(5):
            if images[row][col] is not None:
                mosaic.paste(images[row][col], (col * tile_width, row * tile_height))

    final_name = f"{city}_mosaico_55K.png"
    mosaic.save(final_name)
    print(f"Mosaico guardado como {final_name}")


In [8]:
ciudades=pd.read_csv('Gini_EPH.csv')

In [8]:
len(ciudades)

113

In [9]:
ciudades.head(2)

,Unnamed: 0,Pais,Ciudad,Gini,ciudad pais,latlon,lat,lon
0,0,Austria,Vienna,0.27,Vienna Austria,N 48° 12′ 30''E 16° 22′ 19'',48.208333,16.371944
1,1,Austria,Graz,0.27,Graz Austria,N 47° 4′ 0''E 15° 27′ 0'',47.066667,15.450000


In [20]:
ciudades.columns = ['U', 'Country', 'City', 'Gini', 'latlon', 'Latitude', 'Longitude']

In [38]:
import pandas as pd
import numpy as np
import time
import os

for i in range(len(ciudades)):  # Iterar sobre los primeros elementos
    lat = ciudades.loc[i, "lat"]
    lon = ciudades.loc[i, "lon"]
    #if pd.isna(ciudades.loc[i, "Diferencia"]):  # Solo procesar si "Diferencia" está vacío
    screenshot_name = ciudades.loc[i, "Ciudad"] + " - 40K.png"
    screenshot_name = screenshot_name.replace(":", "_").replace("/", ".")
    try:
        take_image40K(i)
    except Exception as e:
        print(f"⚠️ Error en take_image40K para {lat}, {lon}: {e}")


Screenshot saved to: Vienna - 40K.png
Screenshot saved to: Graz - 40K.png
Screenshot saved to: Linz - 40K.png
Screenshot saved to: Antwerp - 40K.png
Screenshot saved to: Gent - 40K.png
Screenshot saved to: Liege - 40K.png
Screenshot saved to: Toronto - 40K.png
Screenshot saved to: Montreal - 40K.png
Screenshot saved to: Vancouver - 40K.png
Screenshot saved to: Calgary - 40K.png
Screenshot saved to: Winnipeg - 40K.png
Screenshot saved to: Hamilton - 40K.png
Screenshot saved to: London - 40K.png
Screenshot saved to: Halifax - 40K.png
Screenshot saved to: Victoria - 40K.png
Screenshot saved to: Windsor - 40K.png
Screenshot saved to: Saskatoon - 40K.png
Screenshot saved to: Sherbrooke - 40K.png
Screenshot saved to: Paris - 40K.png
Screenshot saved to: Lyon - 40K.png
Screenshot saved to: Toulouse - 40K.png
Screenshot saved to: Strasbourg - 40K.png
Screenshot saved to: Bordeaux - 40K.png
Screenshot saved to: Nantes - 40K.png
Screenshot saved to: Lille - 40K.png
Screenshot saved to: Montpelli

In [13]:
import os
import string

def encontrar_caracter_disponible(archivos):
    caracteres_prohibidos = set("".join(archivos))  # Unir todos los nombres y obtener los caracteres usados
    for caracter in string.punctuation + string.ascii_letters + string.digits:
        if caracter not in caracteres_prohibidos:
            return caracter
    raise ValueError("No hay caracteres disponibles para reemplazar.")

def renombrar_archivos(carpeta):
    archivos = os.listdir(carpeta)
    
    if not archivos:
        print("No hay archivos en la carpeta.")
        return
    
    caracter_nuevo = encontrar_caracter_disponible(archivos)
    
    for archivo in archivos:
        if "'" in archivo:
            nuevo_nombre = archivo.replace("'", caracter_nuevo)
            ruta_vieja = os.path.join(carpeta, archivo)
            ruta_nueva = os.path.join(carpeta, nuevo_nombre)
            os.rename(ruta_vieja, ruta_nueva)
            print(f'Renombrado: "{archivo}" -> "{nuevo_nombre}"')

# Cambia "tu_carpeta" por la ruta de la carpeta donde están los archivos
renombrar_archivos("C:\\Users\\PC\\Downloads\\imagenes2")


Renombrado: "CL_ Libertador Bernardo O'Higgins-Colchagua-Peralillo - 15K.png" -> "CL_ Libertador Bernardo O!Higgins-Colchagua-Peralillo - 15K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Colchagua-Peralillo - 5K.png" -> "CL_ Libertador Bernardo O!Higgins-Colchagua-Peralillo - 5K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Las Cabras-El Manzano - 15K.png" -> "CL_ Libertador Bernardo O!Higgins-Las Cabras-El Manzano - 15K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Las Cabras-El Manzano - 5K.png" -> "CL_ Libertador Bernardo O!Higgins-Las Cabras-El Manzano - 5K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Malloa - 15K.png" -> "CL_ Libertador Bernardo O!Higgins-Malloa - 15K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Malloa - 5K.png" -> "CL_ Libertador Bernardo O!Higgins-Malloa - 5K.png"
Renombrado: "CL_ Libertador Bernardo O'Higgins-Pelequen - 15K.png" -> "CL_ Libertador Bernardo O!Higgins-Pelequen - 15K.png"
Renombrado: "CL_ Libertador Bernardo O'Higg

In [12]:
import pandas as pd
ciudades = pd.DataFrame({
    "City": ["Rocinha", "Retiro", "Oceano", "Amazonas", "Lo Barnechea", "El Alto", "Desierto"],
    "Latitude": ["-22.98607", "-34.58535", "-41.14605", "-5.07532", "-33.33302", "-16.47990", "-37.76729"],
    "Longitude": ["-43.2428", "-58.37679", "-44.13015", "-61.06059", "-70.54222", "-68.20027", "-66.80431"]
})

In [16]:
import time 
import os 

for X in range(len(ciudades)):
    take_image1K(X)
    take_image5K(X)
    take_image10K(X)
    take_image15K(X)

Screenshot saved to: Rocinha - 1K.png
Screenshot saved to: Rocinha - 5K.png
Screenshot saved to: Rocinha - 10K.png
Screenshot saved to: Rocinha - 15K.png
Screenshot saved to: Retiro - 1K.png
Screenshot saved to: Retiro - 5K.png
Screenshot saved to: Retiro - 10K.png
Screenshot saved to: Retiro - 15K.png
Screenshot saved to: Oceano - 1K.png
Screenshot saved to: Oceano - 5K.png
Screenshot saved to: Oceano - 10K.png
Screenshot saved to: Oceano - 15K.png
Screenshot saved to: Amazonas - 1K.png
Screenshot saved to: Amazonas - 5K.png
Screenshot saved to: Amazonas - 10K.png
Screenshot saved to: Amazonas - 15K.png
Screenshot saved to: Lo Barnechea - 1K.png
Screenshot saved to: Lo Barnechea - 5K.png
Screenshot saved to: Lo Barnechea - 10K.png
Screenshot saved to: Lo Barnechea - 15K.png
Screenshot saved to: El Alto - 1K.png
Screenshot saved to: El Alto - 5K.png
Screenshot saved to: El Alto - 10K.png
Screenshot saved to: El Alto - 15K.png
Screenshot saved to: Desierto - 1K.png
Screenshot saved to: 